# 🎓 Educational Tutorial: Validation, Metrics, and Analysis

This notebook serves as a self-contained learning artifact detailing the concepts, equations, and implementation of **validation systems** for land-cover classification and deforestation change detection.

---

## 1. Metrics & Equations

### A. Multiclass Metrics
* **Accuracy**: The ratio of correctly classified pixels/patches to total instances:
  $$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

### B. Binary Deforestation Metrics
* **Sensitivity (Recall)**: Proportion of actual deforested patches correctly identified:
  $$\text{Sensitivity} = \frac{TP}{TP + FN}$$
* **Specificity**: Proportion of stable/no-change patches correctly identified:
  $$\text{Specificity} = \frac{TN}{TN + FP}$$
* **Balanced Accuracy**: The arithmetic mean of sensitivity and specificity:
  $$\text{Balanced Accuracy} = \frac{\text{Sensitivity} + \text{Specificity}}{2}$$
* **Intersection over Union (IoU)**: Measure of overlap between predicted and ground-truth masks:
  $$\text{IoU} = \frac{TP}{TP + FP + FN}$$
* **Dice Coefficient**: Harmonized overlap measurement (equivalent to F1-Score):
  $$\text{Dice} = \frac{2 \times TP}{2 \times TP + FP + FN}$$

In [ ]:
import numpy as np
import pandas as pd
import json
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

print("Validation libraries loaded.")

In [ ]:
class DeforestationPipelineValidator:
    def __init__(self, classes=None, patch_size_m=64.0):
        self.classes = classes or [
            "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
            "Industrial", "Pasture", "PermanentCrop", "Residential",
            "River", "SeaLake"
        ]
        self.patch_size_m = patch_size_m

    def compute_metrics(self, y_true, y_pred):
        tp = int(np.sum((y_pred == 1) & (y_true == 1)))
        fp = int(np.sum((y_pred == 1) & (y_true == 0)))
        fn = int(np.sum((y_pred == 0) & (y_true == 1)))
        tn = int(np.sum((y_pred == 0) & (y_true == 0)))
        
        sens = tp / max(1, tp + fn)
        spec = tn / max(1, tn + fp)
        prec = tp / max(1, tp + fp)
        f1 = 2 * prec * sens / max(1e-6, prec + sens)
        iou = tp / max(1, tp + fp + fn)
        balanced_acc = (sens + spec) / 2.0
        
        return {
            "accuracy": (tp + tn) / max(1, tp + tn + fp + fn),
            "precision": prec,
            "recall": sens,
            "f1_score": f1,
            "iou": iou,
            "specificity": spec,
            "balanced_accuracy": balanced_acc
        }

In [ ]:
y_true = np.array([1, 1, 0, 0, 1, 0, 0, 0])
y_pred = np.array([1, 0, 0, 0, 1, 1, 0, 0])

validator = DeforestationPipelineValidator()
metrics = validator.compute_metrics(y_true, y_pred)
print("Validation metrics:")
print(json.dumps(metrics, indent=2))

# Plot area distribution comparison
plt.figure(figsize=(6, 4))
plt.bar(["True Change", "Pred Change"], [sum(y_true), sum(y_pred)], color=["green", "blue"])
plt.title("True vs Predicted Deforested Patch Counts")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()